# 📋 [OPS08_M03] 무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m03_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M03
- **미션 주제:** 무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 03. 무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"SALUTE 보고 체계를 드론 상황에 맞게 개편한 7개 필수 항목과 시계방향 방위, 상대고도 표현을 적용한 가상 무전 약어표 및 10대 MSEL 시나리오를 작성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 정상 드론 6건과 오인/교란 4건(6:4 비율)이 안배되었는지, 무전 발성 시 혼선이 없는지 점검합니다.

---

### 📑 실습자료 및 과업 지침

무전 약어 및 육안 식별 보고 템플릿(방위, 고도, 기종, 속도) 표준화**

가. 제공 자료 - 드론 보고 7항목 표현 기준(초안) [교육용가상자료]

| **항목** | **표현 방법** | **예시** | **보완 의견(작성)** |
| --- | --- | --- | --- |
| **방위** | 진지 정면을 12시로 한 시계 방향 | 2시 방향 | ( ) |
| **고도** | 지형지물 기준 상대 표현(지면 가까이 / 나무 높이 / 나무 위 / 높음) | 나무 위 | ( ) |
| **기종** | 회전익 / 고정익 / 미상 | 회전익 | ( ) |
| **속도** | 정지비행 / 느림 / 빠름 | 느림 | ( ) |
| **수량** | 기체 수, 불확실하면 '최소 ○기' | 최소 2기 | ( ) |
| **행동** | 접근 / 선회 / 이탈 / 정지 / 급강하 | 접근 중 | ( ) |
| **시각** | 최초 포착 시각(시·분) | 05시 42분 | ( ) |

나. 제공 자료 - 교육용 가상 무전 약어표 (실제 음어·호출부호 사용 금지)

| **약어(가상)** | **뜻** | **사용 예** | **수정·추가(작성)** |
| --- | --- | --- | --- |
| **드경** | 드론 경보(위협 확인) | 관측1, 드경, 2시 나무 위 회전익 1기 | ( ) |
| **확요** | 확인 요망(위협 미확인) | 관측2, 확요, 9시 소리만 | ( ) |
| **복창** | 받은 내용을 그대로 되풀이 | 지휘, 복창, 2시 회전익 1기 | ( ) |
| **은폐** | 즉시 은폐 위치로 이동 | 지휘, 전원 은폐 | ( ) |
| **해제** | 경보 해제 | 지휘, 드경 해제, 인원 보고 | ( ) |
|  |  |  |  |
|  |  |  |  |

다. 제공 자료 - 7항목 보고 문안 예시

| 관측1, 드경. 2시 방향, 나무 위, 회전익 1기, 느림, 접근 중, 05시 42분. 이상.
지휘, 복창. 2시 나무 위 회전익 1기 접근. 전원 은폐. 관측1 계속 추적 보고. |
| --- |

라. 작성 서식 - 상황 주입 시나리오(MSEL) (10건 이상, 정상:비정상 = 6:4)

| **번호** | **가상 시각** | **주입 수단** | **사건 내용(주입 메시지 문안)** | **구분** | **기대 행동** | **담당** |
| --- | --- | --- | --- | --- | --- | --- |
| **1** | 05:40 | 구두 | OP-1 방향에서 멀리 윙윙 소리 (예시) | 정상 | 확요 보고 → 지휘자 복창 (예시) | 관측자 |
| **2** | 05:43 | 카드 | 2시 나무 위 회전익 1기 접근 (예시) | 정상 | 드경 7항목 보고 → 은폐 (예시) | 전원 |
| **3** | 05:47 | 카드 | 9시 방향 새 떼 이동 (예시) | 비정상 | 교차 확인 후 오경보 판정 (예시) | 지휘자 |
| **4** |  |  |  |  |  |  |
| **5** |  |  |  |  |  |  |
| **6** |  |  |  |  |  |  |
| **7** |  |  |  |  |  |  |
| **8** |  |  |  |  |  |  |
| **9** |  |  |  |  |  |  |
| **10** |  |  |  |  |  |  |
| **11** |  |  |  |  |  |  |
| **12** |  |  |  |  |  |  |

| **구성 비율 계산** | 정상 ( )건 : 비정상 ( )건 = ( ) : ( ) → 6:4 충족 여부 ( ) |
| --- | --- |

마. 워크스루(소리 내어 따라가기) 점검표

| **점검 질문** | **문제 발견 사건 번호** | **수정 내용** |
| --- | --- | --- |
| **사건 순서와 간격이 현실적인가(너무 몰리거나 비지 않는가)?** |  |  |
| **모든 사건에서 7항목 보고가 가능한가(빠지는 항목은?)** |  |  |
| **비정상 사건이 너무 쉽게 드러나지 않는가?** |  |  |
| **돌발 변수(통신 두절·센서 오류·부상)가 1건 이상 있는가?** |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 보고 템플릿에 7항목이 모두 있다 | ( ) |
| 약어는 가상 표현만 사용했다 | ( ) |
| 주입 사건이 10건 이상이다 | ( ) |
| 정상/비정상 비율이 6:4다 | ( ) |
| 돌발 변수가 1건 이상 포함되었다 | ( ) |
| 워크스루 지적 사항을 모두 수정했다 | ( ) |

**

